# ffmpeg_data_v9

## Example 1: the cache files are present and parse

In [ ]:
# Example 1: get_cache_path() points at the five JSON tables, each of which parses
import json
import ffmpeg_data_v9

folder = ffmpeg_data_v9.get_cache_path() / "list"
tables = {path.stem: json.loads(path.read_text()) for path in sorted(folder.glob("*.json"))}
assert set(tables) == {"av_option_sets", "codecs", "filters", "formats", "options"}, set(tables)
assert all(len(rows) > 10 for rows in tables.values())
print("Example 1:", {name: len(rows) for name, rows in tables.items()})

## Example 2: the filter table describes scale

In [ ]:
# Example 2: the cached `scale` filter lists the options a user passes (w, h, flags)
import json
import ffmpeg_data_v9

filters = json.loads((ffmpeg_data_v9.get_cache_path() / "list" / "filters.json").read_text())
scale = next(f for f in filters if f["name"] == "scale")
option_names = {option["name"] for option in scale["options"]}
assert {"w", "h", "flags"} <= option_names, sorted(option_names)[:20]
print("Example 2:", scale["name"], sorted(option_names)[:8])

In [ ]:
import json, subprocess
import ffmpeg_data_v9

listing = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True, check=True).stdout
rows = [line.split() for line in listing.splitlines()]
ours = {row[1] for row in rows if len(row) > 2 and "->" in row[2]}
cached = {f["name"] for f in json.loads((ffmpeg_data_v9.get_cache_path() / "list" / "filters.json").read_text())}
missing = sorted(ours - cached)
assert len(ours) > 300 and len(missing) < 0.1 * len(ours), (len(ours), missing[:20])
print("Example 3:", len(ours), "filters in our FFmpeg,", len(missing), "not in the table:", missing[:8])

In [ ]:
import json
import ffmpeg_data_v9

codecs = json.loads((ffmpeg_data_v9.get_cache_path() / "list" / "codecs.json").read_text())
names = {codec["name"] for codec in codecs}
assert {"mpeg4", "aac", "flac", "vp9", "opus", "pcm_s16le"} <= names, sorted(names)[:30]
print("Example 4:", len(names), "codecs; mpeg4/aac/flac/vp9/opus present")

## Example 5: the CLI parser reads the tables

In [ ]:
# Example 5: typed-ffmpeg's command-line parser turns an ffmpeg command back into a graph with these tables
from ffmpeg.compile.compile_cli import compile_as_list, parse

argv = ["ffmpeg", "-i", "in.mp4", "-filter_complex", "[0:v]scale=w=64:h=48[s0]", "-map", "[s0]", "out.mp4"]
graph = parse(" ".join(argv))
rebuilt = compile_as_list(graph)
assert "scale=w=64:h=48" in " ".join(rebuilt) and rebuilt[-1] == "out.mp4", rebuilt
print("Example 5:", " ".join(rebuilt))